# Chat with a Self-Hosted LLM Using Microsoft Agent Framework

This notebook demonstrates how to use the **Microsoft Agent Framework** to build an AI agent that chats with a self-hosted Gemma 4 model running on Azure Container Apps (ACA) with vLLM.

The Agent Framework's `OpenAIChatClient` connects to any OpenAI-compatible endpoint, including the vLLM server deployed in this project.

In [4]:
%pip install agent-framework==1.0.1

  Using cached agent_framework_lab-1.0.0b251024-py3-none-any.whl.metadata (5.7 kB)
  Using cached httpx_sse-0.4.3-py3-none-any.whl.metadata (9.7 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached jsonschema-4.26.0-py3-none-any.whl.metadata (7.6 kB)
  Using cached pydantic_settings-2.13.1-py3-none-any.whl.metadata (3.4 kB)
  Using cached pywin32-311-cp313-cp313-win_amd64.whl.metadata (10 kB)
  Using cached idna-3.11-py3-none-any.whl.metadata (8.4 kB)
  Using cached certifi-2026.2.25-py3-none-any.whl.metadata (2.5 kB)
  Using cached httpcore-1.0.9-py3-none-any.whl.metadata (21 kB)
  Using cached h11-0.16.0-py3-none-any.whl.metadata (8.3 kB)
  Using cached jsonschema_specifications-2025.9.1-py3-none-any.whl.metadata (2.9 kB)
  Using cached referencing-0.37.0-py3-none-any.whl.metadata (2.8 kB)
  Using cached rpds_py-0.30.0-cp313-cp313-win_amd64.whl.metadata (4.2 kB)
  Using cached cffi-2.0.0-cp313-cp313-win_amd64.whl.metadata (2.6 kB)
  Using cached pycparser

## Get the LLM Endpoint

Retrieve the FQDN of the Gemma 4 model deployed on ACA from the Terraform output.

In [2]:
aca_gemma4_31b_it_a100_fqdn = ! terraform output -raw aca_gemma4_31b_it_a100_fqdn
aca_gemma4_31b_it_a100_fqdn = aca_gemma4_31b_it_a100_fqdn.n
print("LLM Endpoint:", aca_gemma4_31b_it_a100_fqdn)

LLM Endpoint: gemma-4-31b-it-a100.blueground-f7716399.swedencentral.azurecontainerapps.io


## 1. Simple Agent — Single Turn

Create an agent backed by the self-hosted Gemma 4 model via `OpenAIChatClient` pointing at the vLLM OpenAI-compatible endpoint.

In [5]:
from agent_framework import Agent
from agent_framework.openai import OpenAIChatClient

client = OpenAIChatClient(
    base_url=f"http://{aca_gemma4_31b_it_a100_fqdn}/v1",
    api_key="EMPTY",
    model="google/gemma-4-31B-it",
)

async with Agent(
    client=client,
    instructions="You are a helpful AI assistant. Be concise and informative.",
) as agent:
    result = await agent.run("What is Azure Container Apps?")
    print(result.text)

**Azure Container Apps (ACA)** is a fully managed serverless platform designed for deploying containerized applications without having to manage the underlying infrastructure (like Kubernetes clusters).

It is built on top of **Azure Kubernetes Service (AKS)** and **KEDA** (Kubernetes Event-driven Autoscaling), but it abstracts the complexity of Kubernetes away from the developer.

### Key Features
*   **Serverless Scaling:** It can scale automatically based on HTTP traffic or events (e.g., messages in a queue). It can even **scale to zero** when not in use to save costs.
*   **Microservices Focused:** Designed specifically for microservices and fragmented architectures.
*   **Dapr Integration:** Comes with built-in support for **Distributed Application Runtime (Dapr)**, which simplifies service-to-service communication, state management, and pub/sub messaging.
*   **Simplified Networking:** Provides built-in ingress (HTTP/TCP) and manages traffic routing and splitting (useful for Blue

## 2. Streaming Response

Use `stream=True` for a token-by-token streaming experience, which is the recommended pattern for production-grade apps.

In [6]:
from agent_framework import Agent
from agent_framework.openai import OpenAIChatClient

client = OpenAIChatClient(
    base_url=f"http://{aca_gemma4_31b_it_a100_fqdn}/v1",
    api_key="EMPTY",
    model="google/gemma-4-31B-it",
)

async with Agent(
    client=client,
    instructions="You are a helpful AI assistant. Be concise and informative.",
) as agent:
    print("Agent: ", end="", flush=True)
    stream = agent.run("Explain Kubernetes in 3 sentences.", stream=True)
    async for chunk in stream:
        if chunk.text:
            print(chunk.text, end="", flush=True)
    print()
    await stream.get_final_response()  # finalize the stream

Agent: Kubernetes is an open-source orchestration platform designed to automate the deployment, scaling, and management of containerized applications. It ensures high availability by monitoring containers and automatically restarting or replacing those that fail. By distributing workloads across a cluster of machines, it optimizes resource usage and allows applications to scale seamlessly based on demand.


## 3. Agent with Tool Calling

Enhance the agent with custom Python functions as tools. The Agent Framework automatically handles the tool-calling loop with the LLM.

In [39]:
from typing import Annotated
from random import randint
from pydantic import Field
from agent_framework import Agent, tool
from agent_framework.openai import OpenAIChatClient


# NOTE: approval_mode="never_require" is for sample brevity.
# Use "always_require" in production for user confirmation before tool execution.
@tool(approval_mode="never_require")
def get_weather(
    location: Annotated[str, Field(description="The location to get the weather for.")],
) -> str:
    """Get the weather for a given location."""
    conditions = ["sunny", "cloudy", "rainy", "stormy"]
    return f"The weather in {location} is {conditions[randint(0, 3)]} with a high of {randint(10, 30)}°C."

# def get_time(
#     timezone: Annotated[str, "The timezone, e.g. 'UTC', 'CET', 'PST'."],
# ) -> str:
#     """Get the current time in a given timezone."""
#     from datetime import datetime
#     return f"The current time in {timezone} is {datetime.now().strftime('%H:%M:%S')} (simulated)."


agent = Agent(
    client=OpenAIChatClient(
        base_url=f"http://{aca_gemma4_31b_it_a100_fqdn}/v1",
        api_key="EMPTY",
        model="google/gemma-4-31B-it",
    ),
    name="WeatherAgent",
    instructions="You are a helpful assistant that can provide weather and restaurant information.",
    tools=[get_weather],
)

result = await agent.run("What's the weather like in Seattle?")
print(f"Agent: {result}")

# print("Agent: ", end="", flush=True)

# stream = await agent.run("What's the weather in Amsterdam and what are today's specials?", stream=True)
# async for chunk in stream:
#     if chunk.text:
#         print(chunk.text, end="", flush=True)
# print()
# await stream.get_final_response()
       

Agent: call:get_weather{location:Seattle}
